# 09 — Interactive AML Dashboard (Dash/Plotly)

Full interactive dashboard with 5 tabs:
1. Executive Summary
2. Financial Impact
3. Network Graph
4. Transaction Deep Dive
5. Node Risk Profiles

Run this notebook and open http://localhost:8050

In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
import warnings
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import dash
from dash import dcc, html, dash_table, Input, Output
import dash_bootstrap_components as dbc
import networkx as nx

from gan_anomaly import Generator, Encoder, anomaly_score as compute_anomaly_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# ── Data Loading & Enrichment ──

transactions = pd.read_parquet("data/transactions_processed.parquet")
node_features_df = pd.read_parquet("data/node_features.parquet")
node_embeddings = pd.read_parquet("results/node_embeddings_scored.parquet")
embeddings = np.load("embeddings/node_embeddings.npy")

# Load GAN for threshold
with open("models/training_meta.json") as f:
    meta = json.load(f)
G_m = Generator(meta["latent_dim"], meta["input_dim"], meta["g_hidden"], meta["n_layers"], meta["activation"]).to(device)
E_m = Encoder(meta["input_dim"], meta["latent_dim"], meta["e_hidden"], meta["n_layers"], meta["activation"]).to(device)
G_m.load_state_dict(torch.load("models/generator.pt", map_location=device, weights_only=True))
E_m.load_state_dict(torch.load("models/encoder.pt", map_location=device, weights_only=True))

X_train = np.load("models/X_train.npy")
train_scores = compute_anomaly_score(torch.tensor(X_train, dtype=torch.float32).to(device), E_m, G_m).cpu().numpy()
threshold_val = float(np.percentile(train_scores, 99))

anomaly_scores = node_embeddings["anomaly_score"].values

# Map risk onto edges
edges_df = transactions.copy()
node_risk_dict = node_embeddings.set_index("id")["risk_score"].to_dict()
node_anomaly_dict = node_embeddings.set_index("id")["is_anomaly"].to_dict()

edges_df["source_risk"] = edges_df["source"].map(node_risk_dict).fillna(0)
edges_df["target_risk"] = edges_df["target"].map(node_risk_dict).fillna(0)
edges_df["edge_risk"] = edges_df[["source_risk", "target_risk"]].max(axis=1)
edges_df["source_anomaly"] = edges_df["source"].map(node_anomaly_dict).fillna(False)
edges_df["target_anomaly"] = edges_df["target"].map(node_anomaly_dict).fillna(False)
edges_df["is_suspicious"] = edges_df["source_anomaly"] | edges_df["target_anomaly"]

node_type_dict = node_features_df.set_index("id")["type"].to_dict()
edges_df["source_type"] = edges_df["source"].map(node_type_dict).fillna(-1).astype(int)
edges_df["target_type"] = edges_df["target"].map(node_type_dict).fillna(-1).astype(int)

# Money flow
outgoing = edges_df.groupby("source").agg({"base_amt": "sum", "tran_id": "count"}).rename(
    columns={"base_amt": "outgoing_amt", "tran_id": "outgoing_count"})
incoming = edges_df.groupby("target").agg({"base_amt": "sum", "tran_id": "count"}).rename(
    columns={"base_amt": "incoming_amt", "tran_id": "incoming_count"})

node_money = node_embeddings[["id", "anomaly_score", "is_anomaly", "risk_score"]].copy()
if "is_sar" in node_embeddings.columns:
    node_money["is_sar"] = node_embeddings["is_sar"]
node_money = node_money.merge(outgoing, left_on="id", right_index=True, how="left")
node_money = node_money.merge(incoming, left_on="id", right_index=True, how="left")
node_money = node_money.fillna(0)
node_money["total_volume"] = node_money["outgoing_amt"] + node_money["incoming_amt"]
node_money["total_transactions"] = node_money["outgoing_count"] + node_money["incoming_count"]
node_money["net_flow"] = node_money["incoming_amt"] - node_money["outgoing_amt"]
node_money = node_money.merge(node_features_df[["id", "type"]], on="id", how="left")

# Financial metrics
n_anomalies = int(node_embeddings["is_anomaly"].sum())
n_normal = len(node_embeddings) - n_anomalies
suspicious_txn_value = float(edges_df[edges_df["is_suspicious"]]["base_amt"].sum())

if "is_sar" in node_embeddings.columns:
    tp = int(((node_embeddings["is_sar"] == 1) & (node_embeddings["is_anomaly"])).sum())
    fp = int(((node_embeddings["is_sar"] == 0) & (node_embeddings["is_anomaly"])).sum())
    fn = int(((node_embeddings["is_sar"] == 1) & (~node_embeddings["is_anomaly"])).sum())
    tn = int(((node_embeddings["is_sar"] == 0) & (~node_embeddings["is_anomaly"])).sum())
else:
    tp, fp = int(n_anomalies * 0.10), n_anomalies - int(n_anomalies * 0.10)
    fn, tn = int(n_normal * 0.01), n_normal - int(n_normal * 0.01)

cfg = {"avg_loss": 0.15, "inv_cost": 500, "fp_cost": 200, "fine_mult": 3.0, "recovery": 0.70}
avg_susp_txn = suspicious_txn_value / max(n_anomalies, 1)
potential_loss = tp * avg_susp_txn * cfg["avg_loss"]
recovered_amount = potential_loss * cfg["recovery"]
normal_vol = float(edges_df[~edges_df["is_suspicious"]]["base_amt"].sum())
loss_undetected = fn * (normal_vol / max(n_normal, 1)) * cfg["avg_loss"]
regulatory_fine_risk = loss_undetected * cfg["fine_mult"]
investigation_cost = n_anomalies * cfg["inv_cost"]
false_positive_cost = fp * cfg["fp_cost"]
total_op_cost = investigation_cost + false_positive_cost
net_savings = recovered_amount - total_op_cost

precision = tp / max(tp + fp, 1)
recall = tp / max(tp + fn, 1)
f1 = 2 * (precision * recall) / max(precision + recall, 1e-9)

print(f"Loaded {len(edges_df):,} transactions, {len(node_embeddings):,} nodes")
print(f"Anomalies: {n_anomalies:,}  Threshold: {threshold_val:.6f}")

Loaded 430,744 transactions, 7,500 nodes
Anomalies: 89  Threshold: 5.807745


In [3]:
# ══════════════════════════════════════════════════════════════
#  Build Dash App
# ══════════════════════════════════════════════════════════════

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.DARKLY])

CLR = dict(blue="#3498db", green="#2ecc71", red="#e74c3c", purple="#9b59b6",
           orange="#e67e22", yellow="#f1c40f", dark="#2c3e50", light="#ecf0f1")
RISK_COLORS = [CLR["green"], CLR["yellow"], CLR["orange"], CLR["red"]]
RISK_LABELS = ["Low", "Medium", "High", "Critical"]

def kpi_card(title, value, color):
    return dbc.Card([dbc.CardBody([
        html.H6(title, className="text-muted mb-1", style={"fontSize": "0.85rem"}),
        html.H3(value, className="mb-0", style={"color": color, "fontWeight": "bold"}),
    ])], className="shadow-sm", style={"borderLeft": f"4px solid {color}"})

# ── TAB 1: Executive Summary ──
risk_cat = pd.cut(node_embeddings["risk_score"], bins=[0, 0.25, 0.5, 0.75, 1.0],
                  labels=RISK_LABELS, include_lowest=True)
risk_counts = risk_cat.value_counts().reindex(RISK_LABELS).fillna(0)

fig_risk_bar = go.Figure(go.Bar(x=RISK_LABELS, y=risk_counts.values, marker_color=RISK_COLORS,
    text=[f"{int(v):,}" for v in risk_counts.values], textposition="outside"))
fig_risk_bar.update_layout(template="plotly_dark", title="Risk Distribution", yaxis_title="Nodes", margin=dict(t=40, b=30))

susp_vol = float(edges_df[edges_df["is_suspicious"]]["base_amt"].sum())
norm_vol = float(edges_df[~edges_df["is_suspicious"]]["base_amt"].sum())
fig_vol_pie = go.Figure(go.Pie(labels=["Normal", "Suspicious"], values=[norm_vol, susp_vol],
    marker_colors=[CLR["blue"], CLR["red"]], hole=0.45, textinfo="label+percent"))
fig_vol_pie.update_layout(template="plotly_dark", title="Volume: Normal vs Suspicious", margin=dict(t=40, b=10))

top10 = node_money.nlargest(10, "risk_score")
fig_top10 = go.Figure(go.Bar(
    y=[f"{r['id'][:10]}… ({r['risk_score']:.2f})" for _, r in top10.iterrows()],
    x=top10["total_volume"], orientation="h",
    marker_color=px.colors.sample_colorscale("Reds", top10["risk_score"].values)))
fig_top10.update_layout(template="plotly_dark", title="Top 10 Risk Nodes", xaxis_title="Volume ($)",
                        yaxis=dict(autorange="reversed"), margin=dict(t=40, b=30, l=160))

sorted_scores = np.sort(anomaly_scores)
fig_score_curve = go.Figure()
fig_score_curve.add_trace(go.Scatter(x=list(range(len(sorted_scores))), y=sorted_scores,
    fill="tozeroy", fillcolor="rgba(52,152,219,0.2)", line=dict(color=CLR["blue"], width=2)))
fig_score_curve.add_hline(y=threshold_val, line_dash="dash", line_color="red",
                          annotation_text=f"Threshold {threshold_val:.6f}")
fig_score_curve.update_layout(template="plotly_dark", title="Anomaly Score Curve",
                              xaxis_title="Nodes (sorted)", yaxis_title="Score", margin=dict(t=40, b=30))

tab1 = dbc.Container([
    dbc.Row([
        dbc.Col(kpi_card("Total Transactions", f"{len(edges_df):,}", CLR["blue"]), md=3),
        dbc.Col(kpi_card("Total Volume", f'${edges_df["base_amt"].sum():,.0f}', CLR["green"]), md=3),
        dbc.Col(kpi_card("Anomalies", f"{n_anomalies:,}", CLR["red"]), md=3),
        dbc.Col(kpi_card("Detection Rate", f'{100*node_embeddings["is_anomaly"].mean():.1f}%', CLR["purple"]), md=3),
    ], className="mb-3 g-3"),
    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_risk_bar), md=6),
        dbc.Col(dcc.Graph(figure=fig_vol_pie), md=6),
    ], className="mb-3"),
    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_top10), md=6),
        dbc.Col(dcc.Graph(figure=fig_score_curve), md=6),
    ]),
], fluid=True)

# ── TAB 2: Financial Impact ──
cm = np.array([[tn, fp], [fn, tp]])
fig_cm = px.imshow(cm, text_auto=True, x=["Pred Normal", "Pred Anomaly"],
    y=["Actual Normal", "Actual AML"], color_continuous_scale="RdYlGn_r")
fig_cm.update_layout(template="plotly_dark", title="Detection Matrix", margin=dict(t=40, b=30))

fig_waterfall = go.Figure(go.Waterfall(
    x=["Suspicious\nValue", "Loss\nAvoided", "Recovered", "Investigation\nCost", "FP Cost", "Net Savings"],
    y=[suspicious_txn_value, -potential_loss, recovered_amount, -investigation_cost, -false_positive_cost, net_savings],
    measure=["absolute", "relative", "relative", "relative", "relative", "total"],
    increasing_marker_color=CLR["green"], decreasing_marker_color=CLR["red"], totals_marker_color=CLR["purple"],
    texttemplate="$%{y:,.0f}", textposition="outside"))
fig_waterfall.update_layout(template="plotly_dark", title="Loss vs Savings", yaxis_title="$", margin=dict(t=40, b=30))

fig_gauges = make_subplots(rows=1, cols=3, specs=[[{"type": "indicator"}]*3],
                           subplot_titles=["Precision", "Recall", "F1"])
for i, (val, clr) in enumerate([(precision, CLR["blue"]), (recall, CLR["green"]), (f1, CLR["purple"])], 1):
    fig_gauges.add_trace(go.Indicator(mode="gauge+number", value=val*100, number_suffix="%",
        gauge=dict(axis=dict(range=[0, 100]), bar_color=clr)), row=1, col=i)
fig_gauges.update_layout(template="plotly_dark", height=280, margin=dict(t=40, b=10))

tab2 = dbc.Container([
    dbc.Row([
        dbc.Col(kpi_card("Recovered", f"${recovered_amount:,.0f}", CLR["green"]), md=3),
        dbc.Col(kpi_card("Op Cost", f"${total_op_cost:,.0f}", CLR["orange"]), md=3),
        dbc.Col(kpi_card("Net Savings", f"${net_savings:,.0f}", CLR["green"] if net_savings >= 0 else CLR["red"]), md=3),
        dbc.Col(kpi_card("Reg Risk", f"${regulatory_fine_risk:,.0f}", CLR["red"]), md=3),
    ], className="mb-3 g-3"),
    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_cm), md=5),
        dbc.Col(dcc.Graph(figure=fig_waterfall), md=7),
    ], className="mb-3"),
    dbc.Row([dbc.Col(dcc.Graph(figure=fig_gauges), md=12)]),
], fluid=True)

# ── TAB 3: Network Graph ──
_vol_dict = node_money.set_index("id")["total_volume"].to_dict()
_txn_dict = node_money.set_index("id")["total_transactions"].to_dict()

def build_network_figure(filter_mode="all", max_nodes=300):
    edges_sorted = edges_df.sort_values(["edge_risk", "base_amt"], ascending=[False, False])
    if filter_mode == "suspicious":
        edges_subset = edges_sorted[edges_sorted["is_suspicious"]].head(max_nodes * 3)
    elif filter_mode == "high_risk":
        hr_nodes = set(node_embeddings[node_embeddings["risk_score"] > 0.75]["id"])
        edges_subset = edges_sorted[
            edges_sorted["source"].isin(hr_nodes) | edges_sorted["target"].isin(hr_nodes)
        ].head(max_nodes * 3)
    else:
        edges_subset = edges_sorted.head(max_nodes * 3)

    selected = set(edges_subset["source"].tolist() + edges_subset["target"].tolist())
    if len(selected) > max_nodes:
        top = node_embeddings[node_embeddings["id"].isin(selected)].nlargest(max_nodes, "risk_score")["id"]
        selected = set(top)
        edges_subset = edges_subset[
            edges_subset["source"].isin(selected) & edges_subset["target"].isin(selected)]

    G_nx = nx.DiGraph()
    for _, r in edges_subset.iterrows():
        G_nx.add_edge(r["source"], r["target"])
    if G_nx.number_of_nodes() == 0:
        fig = go.Figure(); fig.add_annotation(text="No nodes", xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(template="plotly_dark"); return fig

    pos = nx.spring_layout(G_nx, k=3/np.sqrt(G_nx.number_of_nodes()), iterations=50, seed=42)
    edge_x, edge_y = [], []
    for u, v in G_nx.edges():
        x0, y0 = pos[u]; x1, y1 = pos[v]
        edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

    node_x = [pos[n][0] for n in G_nx.nodes()]
    node_y = [pos[n][1] for n in G_nx.nodes()]
    node_risks = [node_risk_dict.get(n, 0) for n in G_nx.nodes()]
    node_vols = [_vol_dict.get(n, 0) for n in G_nx.nodes()]
    vol_max = max(node_vols) if node_vols else 1
    sizes = [6 + 20 * (v / vol_max) for v in node_vols]
    hovers = [f"ID: {n[:16]}<br>Risk: {node_risk_dict.get(n,0):.4f}<br>Vol: ${_vol_dict.get(n,0):,.0f}" for n in G_nx.nodes()]

    fig = go.Figure(data=[
        go.Scatter(x=edge_x, y=edge_y, mode="lines", line=dict(width=0.5, color="rgba(150,150,150,0.3)"), hoverinfo="none"),
        go.Scatter(x=node_x, y=node_y, mode="markers", marker=dict(size=sizes, color=node_risks,
            colorscale="RdYlGn_r", cmin=0, cmax=1, colorbar=dict(title="Risk"), line=dict(width=0.5, color="white")),
            text=hovers, hoverinfo="text")
    ])
    fig.update_layout(template="plotly_dark", title=f"Network ({G_nx.number_of_nodes()} nodes)",
        showlegend=False, hovermode="closest",
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        margin=dict(t=40, b=10, l=10, r=10), height=600)
    return fig

tab3 = dbc.Container([
    dbc.Row([dbc.Col([
        html.Label("Filter:", className="me-2"),
        dcc.Dropdown(id="network-filter", options=[
            {"label": "All Nodes", "value": "all"},
            {"label": "Suspicious Only", "value": "suspicious"},
            {"label": "High-Risk", "value": "high_risk"}],
            value="all", clearable=False, style={"color": "#000"}),
    ], md=4)], className="mb-3"),
    dbc.Row([dbc.Col(dcc.Graph(id="network-graph", figure=build_network_figure("all")), md=12)]),
], fluid=True)

# ── TAB 4: Transaction Deep Dive ──
fig_amt_hist = px.histogram(edges_df, x="base_amt", nbins=80, color_discrete_sequence=[CLR["blue"]])
fig_amt_hist.update_layout(template="plotly_dark", title="Amount Distribution", yaxis_type="log", margin=dict(t=40, b=30))

samp = edges_df.sample(min(5000, len(edges_df)), random_state=42)
fig_amt_risk = px.scatter(samp, x="base_amt", y="edge_risk", color="edge_risk", color_continuous_scale="RdYlGn_r")
fig_amt_risk.update_layout(template="plotly_dark", title="Amount vs Risk", margin=dict(t=40, b=30))

heatmap_data = edges_df.pivot_table(values="edge_risk", index="source_type", columns="target_type", aggfunc="mean").fillna(0)
fig_heatmap = px.imshow(heatmap_data, color_continuous_scale="RdYlGn_r", text_auto=".3f",
    x=[f"Type {c}" for c in heatmap_data.columns], y=[f"Type {i}" for i in heatmap_data.index])
fig_heatmap.update_layout(template="plotly_dark", title="Avg Risk by Node-Type Pair", margin=dict(t=40, b=30))

tab4 = dbc.Container([
    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_amt_hist), md=6),
        dbc.Col(dcc.Graph(figure=fig_amt_risk), md=6),
    ], className="mb-3"),
    dbc.Row([dbc.Col(dcc.Graph(figure=fig_heatmap), md=12)]),
], fluid=True)

# ── TAB 5: Node Risk Profiles ──
fig_risk_hist = go.Figure()
fig_risk_hist.add_trace(go.Histogram(x=node_money["risk_score"], nbinsx=60, marker_color=CLR["blue"], opacity=0.75))
for p, clr in [(50, "green"), (75, "yellow"), (90, "orange"), (95, "red"), (99, "darkred")]:
    val = float(np.percentile(node_money["risk_score"], p))
    fig_risk_hist.add_vline(x=val, line_dash="dash", line_color=clr, annotation_text=f"P{p}: {val:.3f}")
fig_risk_hist.update_layout(template="plotly_dark", title="Risk Score Distribution", margin=dict(t=40, b=30))

fig_vol_risk = px.scatter(node_money, x="total_volume", y="risk_score", color="is_anomaly",
    color_discrete_map={True: CLR["red"], False: CLR["blue"]})
fig_vol_risk.update_layout(template="plotly_dark", title="Volume vs Risk", margin=dict(t=40, b=30))

top20 = node_money.nlargest(20, "risk_score")[["id", "risk_score", "total_volume", "total_transactions", "is_anomaly"]].copy()
top20["total_volume"] = top20["total_volume"].apply(lambda x: f"${x:,.0f}")
top20["risk_score"] = top20["risk_score"].round(4)

tab5 = dbc.Container([
    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_risk_hist), md=6),
        dbc.Col(dcc.Graph(figure=fig_vol_risk), md=6),
    ], className="mb-3"),
    dbc.Row([dbc.Col([
        html.H6("Top 20 Highest Risk Nodes", className="text-center mb-2"),
        dash_table.DataTable(
            data=top20.to_dict("records"),
            columns=[{"name": c, "id": c} for c in top20.columns],
            sort_action="native", page_size=10,
            style_header={"backgroundColor": "#2c3e50", "color": "white", "fontWeight": "bold"},
            style_cell={"backgroundColor": "#1a1a2e", "color": "white", "fontSize": "12px", "padding": "6px"},
        ),
    ], md=12)]),
], fluid=True)

# ── App Layout ──
app.layout = dbc.Container([
    dbc.Row([dbc.Col([
        html.H2("AML Detection — Interactive Dashboard", className="text-center mt-3 mb-1"),
        html.Div([
            dbc.Badge("PyTorch + GraphSAGE", color="info", className="me-2"),
            html.Span(f"{len(node_embeddings):,} nodes | {len(edges_df):,} txns | {n_anomalies:,} anomalies",
                      className="text-muted"),
        ], className="text-center mb-3"),
    ], md=12)]),
    dbc.Tabs([
        dbc.Tab(tab1, label="Executive Summary"),
        dbc.Tab(tab2, label="Financial Impact"),
        dbc.Tab(tab3, label="Network Graph"),
        dbc.Tab(tab4, label="Transaction Deep Dive"),
        dbc.Tab(tab5, label="Node Risk Profiles"),
    ], active_tab="tab-0"),
], fluid=True)

@app.callback(Output("network-graph", "figure"), Input("network-filter", "value"))
def update_network(filter_mode):
    return build_network_figure(filter_mode)

print("Dash app ready.")

Dash app ready.


In [4]:
# Launch the dashboard
app.run(jupyter_mode="inline", port=8050)

In [5]:
# ── GPU Cleanup — free VRAM for next notebook ──
import gc
for v in ["G_m", "E_m"]:
    if v in dir():
        exec(f"del {v}")
torch.cuda.empty_cache(); gc.collect()
print(f"GPU freed: {torch.cuda.memory_allocated()/1e6:.1f} MB allocated")

GPU freed: 8.5 MB allocated
